# Optimization Exploration using Simulated Annealing

This notebook explores the application of optimization techniques for improving model performance through parameter tuning and feature selection.

Due to computational constraints and convergence challenges, results were inconsistent and not integrated into the final model pipeline.

In [ ]:

# ===================================================================
# FINAL SCRIPT: EEG Classification with Simulated Annealing (SA) 

!pip install catboost simanneal -q

import numpy as np, pandas as pd, os, re, time, random, json, joblib, warnings
warnings.filterwarnings("ignore")
from google.colab import drive
from datetime import datetime
from simanneal import Annealer

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

KAGGLE_DATASET_PATH = '/content/drive/MyDrive/Kaggle Dataset'
RESULTS_BASE_PATH = '/content/drive/MyDrive/Kaggle Dataset/Results/ResultsAfterOptimization'

class FeatureAnnealer(Annealer):
    def __init__(self, state, X_train, y_train, X_val, y_val, model, time_limit_sec=180, patience=80):
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.model = model
        self.cache = {}
        self.best_energy_seen = np.inf
        self.no_improve = 0
        self.patience = patience
        self.time_limit_sec = time_limit_sec
        self.start_time = time.monotonic()
        super(FeatureAnnealer, self).__init__(state)

    def move(self):
        i = random.randint(0, len(self.state) - 1)
        self.state[i] = 1 - self.state[i]

    def energy(self):
        if self.time_limit_sec and (time.monotonic() - self.start_time) > self.time_limit_sec:
            self.user_exit = True
        mask_key = tuple(self.state)
        if mask_key in self.cache:
            acc = self.cache[mask_key]
            e = 1.0 - acc
            if e < self.best_energy_seen:
                self.best_energy_seen, self.no_improve = e, 0
            else:
                self.no_improve += 1
            if self.patience and self.no_improve >= self.patience:
                self.user_exit = True
            return e

        binary_mask = np.fromiter(mask_key, dtype=np.uint8).astype(bool)
        if not np.any(binary_mask):
            e = 1.0
            self.cache[mask_key] = 0.0
            self.no_improve += 1
            if self.patience and self.no_improve >= self.patience:
                self.user_exit = True
            return e

        Xtr = self.X_train[:, binary_mask]
        Xva = self.X_val[:, binary_mask]
        self.model.fit(Xtr, self.y_train)
        acc = self.model.score(Xva, self.y_val)
        self.cache[mask_key] = acc
        e = 1.0 - acc
        if e < self.best_energy_seen:
            self.best_energy_seen, self.no_improve = e, 0
        else:
            self.no_improve += 1
        if self.patience and self.no_improve >= self.patience:
            self.user_exit = True
        return e

def main():
    drive.mount('/content/drive', force_remount=True)
    total_start_time = time.monotonic()

    print("\nPROCESS UPDATE: Setting up Google Colab file paths...")
    optimization_technique_name = "SA"
    base_results_dir = os.path.join(RESULTS_BASE_PATH, optimization_technique_name)
    os.makedirs(base_results_dir, exist_ok=True)
    print(f"✅ Main results directory is set to: {base_results_dir}")

    datasets = {
        "Relax": os.path.join(KAGGLE_DATASET_PATH, "Relax"),
        "Arithmetic": os.path.join(KAGGLE_DATASET_PATH, "Arithmetic"),
        "Mirror": os.path.join(KAGGLE_DATASET_PATH, "Mirror_image"),
        "Stroop": os.path.join(KAGGLE_DATASET_PATH, "Stroop"),
    }

    print("\nPROCESS UPDATE: Loading and preparing data for all subjects...")
    all_data_list, groups_list = [], []
    for state, path in datasets.items():
        if not os.path.exists(path):
            print(f"   -> ⚠️ WARNING: Path for '{state}' does not exist, skipping: {path}")
            continue
        files = [f for f in os.listdir(path) if f.endswith('.csv')]
        for file in files:
            m = re.search(r"sub_(\d+)", file)
            if not m:
                continue
            subject_id = int(m.group(1))
            temp_df = pd.read_csv(os.path.join(path, file), index_col=0)
            temp_df['state'] = state
            all_data_list.append(temp_df)
            groups_list.extend([subject_id] * len(temp_df))

    if not all_data_list:
        raise ValueError("FATAL ERROR: No data files were loaded. Check paths.")

    df = pd.concat(all_data_list, ignore_index=True)
    groups = np.array(groups_list)
    X = df.drop(columns=['state'])
    y_raw = df['state']

    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    class_names = list(le.classes_)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print("✅ Data loading, label encoding, and scaling complete.")

    print("\nPROCESS UPDATE: Configuring SA with a single hold-out validation set for speed...")
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
    train_indices, val_indices = next(splitter.split(X_scaled, y, groups=groups))
    X_train, y_train = X_scaled[train_indices], y[train_indices]
    X_val, y_val = X_scaled[val_indices], y[val_indices]

    fitness_model = RidgeClassifier()
    np.random.seed(42); random.seed(42)
    initial_state = np.random.randint(0, 2, size=X_scaled.shape[1]).tolist()

    annealer = FeatureAnnealer(
        initial_state, X_train, y_train, X_val, y_val, fitness_model,
        time_limit_sec=180, patience=80
    )
    annealer.steps = 250
    annealer.Tmax, annealer.Tmin = 1.0, 0.01

    print(f"   - Fitness model: RidgeClassifier (fast proxy)")
    print(f"   - Iterations (max steps): {annealer.steps}")
    print(f"   - Time limit: {annealer.time_limit_sec}s, Patience: {annealer.patience}")

    best_state, best_energy = annealer.anneal()
    print("\n✅ Simulated Annealing complete.")
    print(f"   - Best proxy accuracy: {1.0 - best_energy:.4f}")
    print(f"   - Unique subsets evaluated: {len(annealer.cache)}")

    selected_features_mask = np.array(best_state).astype(bool)
    X_selected = X_scaled[:, selected_features_mask]
    print(f"✅ Selected {np.sum(selected_features_mask)} / {X_scaled.shape[1]} features.")

    print("\nPROCESS UPDATE: Starting final, robust model training and evaluation loop...")
    models = {
        'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1),
        'Linear SVC': SVC(kernel='linear', probability=True, random_state=42),
        'CatBoost': CatBoostClassifier(random_state=42, verbose=0),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        'KNN': KNeighborsClassifier(n_jobs=-1),
        'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
        'NonLinearSVC-RBF': SVC(kernel='rbf', probability=True, random_state=42)
    }

    final_cv = GroupKFold(n_splits=5)
    final_splits = list(final_cv.split(X_selected, y, groups=groups))
    results_summary = []

    for model_name, model in models.items():
        print(f"\n--- Training {model_name} ---")
        model_results_dir = os.path.join(base_results_dir, model_name.replace(" ", "_"))
        os.makedirs(model_results_dir, exist_ok=True)

        y_pred_list, y_proba_list, y_true_list = [], [], []

        for train_idx, test_idx in final_splits:
            Xtr, Xte = X_selected[train_idx], X_selected[test_idx]
            ytr, yte = y[train_idx], y[test_idx]
            model.fit(Xtr, ytr)
            y_pred_list.extend(model.predict(Xte))
            if hasattr(model, "predict_proba"):
                y_proba_list.extend(model.predict_proba(Xte))
            else:
                # fallback: decision_function -> pseudo-proba via softmax
                scores = model.decision_function(Xte)
                if scores.ndim == 1:
                    scores = np.vstack([-scores, scores]).T
                exp = np.exp(scores - scores.max(axis=1, keepdims=True))
                probs = exp / exp.sum(axis=1, keepdims=True)
                y_proba_list.extend(probs)
            y_true_list.extend(yte)

        y_pred = np.array(y_pred_list)
        y_proba = np.array(y_proba_list)
        y_true = np.array(y_true_list)

        metrics = {
            'Model': model_name,
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, average='weighted'),
            'Recall': recall_score(y_true, y_pred, average='weighted'),
            'F1-Score': f1_score(y_true, y_pred, average='weighted'),
            'AUC': roc_auc_score(y_true, y_proba, multi_class='ovr')
        }
        results_summary.append(metrics)
        print(classification_report(y_true, y_pred, target_names=class_names))

        joblib.dump(model.fit(X_selected, y), os.path.join(model_results_dir, f'{model_name.replace(" ", "_")}_model.joblib'))

    summary_df = pd.DataFrame(results_summary)
    print("\n" + "="*80 + "\n--- FINAL PERFORMANCE SUMMARY ---\n" + "="*80)
    print(summary_df.to_string(index=False))
    summary_df.to_csv(os.path.join(base_results_dir, 'all_models_summary.csv'), index=False)

    total_end_time = time.monotonic()
    print(f"\nTotal execution time: {(total_end_time - total_start_time) / 60:.2f} minutes.")
    print("🎉 All processes finished successfully!")

if __name__ == "__main__":
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 9.1 MB/s eta 0:00:00
Mounted at /content/drive

PROCESS UPDATE: Setting up Google Colab file paths...
✅ Main results directory is set to: /content/drive/MyDrive/Kaggle Dataset/Results/ResultsAfterOptimization/SA

PROCESS UPDATE: Loading and preparing data for all subjects...
✅ Data loading, label encoding, and scaling complete.

PROCESS UPDATE: Configuring SA with a single hold-out validation set for speed...
   - Fitness model: RidgeClassifier (fast proxy)
   - Iterations (max steps): 250
   - Time limit: 180s, Patience: 80


 Temperature        Energy    Accept   Improve     Elapsed   Remaining



✅ Simulated Annealing complete.
   - Best proxy accuracy: 0.2996
   - Unique subsets evaluated: 95
✅ Selected 18 / 32 features.

PROCESS UPDATE: Starting final, robust model training and evaluation loop...

--- Training XGBoost ---
              precision    recall  f1-score   support

  Arithmetic       0.28      0.25      0.26    384000
      Mirror       0.27      0.24      0.26    384000
       Relax       0.33      0.31      0.32    384000
      Stroop       0.31      0.40      0.35    384000

    accuracy                           0.30   1536000
   macro avg       0.30      0.30      0.30   1536000
weighted avg       0.30      0.30      0.30   1536000


--- Training Linear SVC ---
